# Stage 2 Colab A100 smoke executor

This notebook is deliberately a thin runner for the committed source package. It does not alter experiment factors or package runtime evidence into the source transport.

In [ ]:
import subprocess
gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
).strip().splitlines()[0]
if "A100" not in gpu:
    raise RuntimeError(f"Stage 2 requires A100, found {gpu}")
print(gpu)

In [ ]:
from hashlib import sha256
import json
from pathlib import Path, PurePosixPath
import shutil
import zipfile

source_zip = Path('/content/stage2_source.zip')
source_unpack = Path('/content/stage2_source_unpack')
repo = Path('/content/stage2_repo')
bundle_verify_repo = Path('/content/stage2_bundle_verify')
for path in (source_unpack, repo, bundle_verify_repo):
    if path.exists():
        shutil.rmtree(path)
with zipfile.ZipFile(source_zip) as archive:
    names = archive.namelist()
    if set(names) != {'stage2_source.bundle', 'source_metadata.json'} or len(names) != len(set(names)):
        raise RuntimeError('unexpected source archive members')
    if any(PurePosixPath(name).is_absolute() or '..' in PurePosixPath(name).parts or '\\' in name for name in names):
        raise RuntimeError('unsafe source archive path')
    source_unpack.mkdir()
    for name in names:
        (source_unpack / name).write_bytes(archive.read(name))
metadata = json.loads((source_unpack / 'source_metadata.json').read_text(encoding='utf-8'))
bundle = source_unpack / 'stage2_source.bundle'
if sha256(bundle.read_bytes()).hexdigest() != metadata['bundle_sha256']:
    raise RuntimeError('source bundle hash does not match metadata')
subprocess.run(['git', 'init', str(bundle_verify_repo)], check=True, capture_output=True, text=True)
if subprocess.run(['git', '-C', str(bundle_verify_repo), 'bundle', 'verify', str(bundle)], text=True, capture_output=True).returncode != 0:
    raise RuntimeError('source git bundle verification failed')
subprocess.run(['git', 'clone', str(bundle), str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', metadata['git']['commit']], check=True)
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
if commit != metadata['git']['commit']:
    raise RuntimeError('cloned source commit does not match metadata')
if subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain'], text=True).strip():
    raise RuntimeError('cloned source tree is dirty')
print(commit)

In [ ]:
%cd /content/stage2_repo
!python -m pip install --upgrade pip
!python -m pip install torch==2.11.0 --index-url https://download.pytorch.org/whl/cu128
!python -m pip install -r requirements.txt
!python -m pip freeze > /content/stage2_pip_freeze.txt
!python -m unittest discover -s tests -v

In [ ]:
!python monitor_stage2_job.py --events ties_results/.stage2_monitor/colab_a100_run1.events.jsonl --watch ties_results/stage2_smoke/colab_a100_run1 -- python run_stage2_smoke.py --mode primary --environment colab_a100 --protocol docs/paper_rebuild/FROZEN_EXPERIMENT_PROTOCOL.md --output-dir ties_results/stage2_smoke/colab_a100_run1 --fresh
!python validate_stage2_smoke.py --root ties_results/stage2_smoke/colab_a100_run1 --conditions standard_lora full_sr class_prior_reweight --canonical-dir ties_results/canonical_v1

In [ ]:
!python monitor_stage2_job.py --events ties_results/.stage2_monitor/colab_a100_repeat_full_sr.events.jsonl --watch ties_results/stage2_smoke/colab_a100_repeat_full_sr -- python run_stage2_smoke.py --mode repeat_full_sr --environment colab_a100 --protocol docs/paper_rebuild/FROZEN_EXPERIMENT_PROTOCOL.md --output-dir ties_results/stage2_smoke/colab_a100_repeat_full_sr --fresh
!python validate_stage2_smoke.py --root ties_results/stage2_smoke/colab_a100_run1 --conditions standard_lora full_sr class_prior_reweight --canonical-dir ties_results/canonical_v1 --compare-repeat ties_results/stage2_smoke/colab_a100_repeat_full_sr
!python freeze_stage2_environment.py --protocol docs/paper_rebuild/FROZEN_EXPERIMENT_PROTOCOL.md --smoke-root ties_results/stage2_smoke/colab_a100_run1 --source-archive /content/stage2_source.zip --commands ties_results/stage2_smoke/colab_a100_run1/commands.json --output-dir ties_results/stage2_smoke/freeze_bundle --fresh

In [ ]:
from pathlib import Path
import zipfile

evidence = Path('/content/stage2_a100_evidence.zip')
if evidence.exists():
    evidence.unlink()
roots = [Path('ties_results/stage2_smoke'), Path('ties_results/.stage2_monitor')]
with zipfile.ZipFile(evidence, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
    for root in roots:
        if not root.exists():
            continue
        for path in sorted(root.rglob('*')):
            if path.is_file() and path.suffix != '.pt':
                archive.write(path, path.as_posix())
print(evidence)